# 04 — Noise Injection and Text Preprocessing

This notebook constructs controlled noisy and cleaned-noisy variants of the
manually validated and weak-labelled 100-example pilot dataset.

The purpose of this stage is to evaluate how different text-cleaning and
preprocessing strategies affect the preservation of linguistic cues associated
with potential disempowerment.

The experiment uses three text conditions:

1. **Clean** — the original assistant response.
2. **Noisy** — the original response after controlled synthetic noise is added.
3. **Cleaned-noisy** — the noisy response after a specified preprocessing
   configuration is applied.

Noise injection is deterministic and reproducible. Each noisy example is
derived from the same clean source text using a fixed random seed and an
explicit noise configuration.

This stage does not train or evaluate a classifier. Its role is to generate
controlled text variants that will later be used in the modelling and
robustness experiments.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/workspaces/irp-disempowerment-nlp")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RANDOM_SEED = 42

labelled_pilot_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_disempowerment_weak_v5_pilot_100.csv"
)

print("Project root:", PROJECT_ROOT)
print("Labelled pilot path:", labelled_pilot_path)
print("Pilot exists:", labelled_pilot_path.exists())

In [ ]:
pilot_df = pd.read_csv(labelled_pilot_path)

print("Pilot rows:", len(pilot_df))
print(
    "Unique source conversations:",
    pilot_df["source_index"].nunique(),
)
print(
    "Weak-label versions:",
    pilot_df["weak_label_version"].value_counts().to_dict(),
)

In [ ]:
assert len(pilot_df) == 100
assert pilot_df["source_index"].nunique() == 100

assert pilot_df["user_text"].notna().all()
assert pilot_df["assistant_text"].notna().all()

assert (
    pilot_df["weak_label_version"]
    .eq("disempowerment_weak_v5")
    .all()
)

assert int(pilot_df["directive_advice"].sum()) == 6
assert int(pilot_df["sycophantic_validation"].sum()) == 0
assert int(pilot_df["overconfident_judgement"].sum()) == 0

print("Frozen v5 pilot integrity checks passed.")

## Controlled noise specification

Noise is injected only into `assistant_text`.

The user message, weak labels, source identifiers, manual relevance annotations,
and weak-label evidence fields remain unchanged. This isolates the effect of
text corruption on the linguistic cues used for downstream disempowerment
classification.

Noise generation is deterministic. Each transformation uses the project random
seed together with the example identifier so that the same source example and
noise configuration always produce the same noisy text.

The pilot uses a single moderate severity level. The aim is to create realistic
text corruption without deliberately changing the semantic meaning of the
assistant response.

### 1. Casing variation

Randomly alter the case of approximately 10% of eligible alphabetic words.

A selected word may be converted to uppercase or lowercase.

Examples:

- `You should talk to your manager.`
- `You SHOULD talk to your manager.`
- `you should talk to your manager.`

This tests sensitivity to inconsistent capitalization while preserving lexical
content.

### 2. Repeated punctuation

Approximately 20% of eligible sentence-ending punctuation marks are expanded
to repeated forms.

Examples:

- `I understand.`
- `I understand...`
- `Are you sure?`
- `Are you sure???`

Only existing `.`, `!`, and `?` punctuation is modified. New semantic content
is not introduced.

### 3. Whitespace corruption

Approximately 10% of eligible whitespace boundaries are modified.

Transformations may include:

- duplicated spaces;
- removal of a space between suitable tokens;
- insertion of an additional space around punctuation.

Examples:

- `You should speak to them.`
- `You  should speak to them.`
- `You should speak  to them.`

Whitespace corruption must not remove or reorder words.

### 4. Character-level typos

Approximately 3% of eligible words receive one character-level perturbation.

Permitted transformations are:

- swap two adjacent internal alphabetic characters;
- delete one internal alphabetic character;
- duplicate one internal alphabetic character.

Words shorter than four alphabetic characters are excluded.

Examples:

- `manager` → `mangaer`
- `support` → `suport`
- `talking` → `talkking`

Only one character perturbation is applied to a selected word.

### 5. Word deletion

Approximately 2% of eligible words are removed.

The deletion rate is deliberately low because removing too many words could
change the meaning of the response or destroy the weak-label cue itself.

The transformation must:

- retain at least 90% of the original word tokens;
- never remove all text;
- preserve the original order of all remaining words.

This condition represents omissions that may occur in noisy text while keeping
semantic disruption limited.

### 6. Filler/disfluency insertion

A filler token is inserted at approximately 3% of eligible word boundaries.

Permitted fillers are drawn from a fixed list such as:

- `um`
- `uh`
- `well`
- `you know`

Examples:

- `You should talk to your manager.`
- `Well, you should talk to your manager.`
- `You should, um, talk to your manager.`

Inserted fillers do not replace original words.

## Experimental noise conditions

Each pilot example will retain its original clean assistant response and will
be used to create the following controlled variants:

1. `clean`
2. `casing`
3. `punctuation`
4. `whitespace`
5. `typo`
6. `word_deletion`
7. `filler`
8. `mixed`

The `mixed` condition applies all six noise families at the same moderate
severity using deterministic randomisation.

Single-noise conditions allow the effect of each noise type to be examined
independently, while the mixed condition represents a more challenging
combined-noise scenario.

These transformations are synthetic experimental perturbations. They are not
intended to estimate the natural frequency of these noise types in
LMSYS-Chat-1M.

In [ ]:
import re

from src.noise import (
    NOISE_VERSION,
    NOISE_CONDITIONS,
    apply_noise_condition,
    generate_noise_variants,
)


assert NOISE_VERSION == "noise_v1"

assert NOISE_CONDITIONS == (
    "clean",
    "casing",
    "punctuation",
    "whitespace",
    "typo",
    "word_deletion",
    "filler",
    "mixed",
)


synthetic_text = (
    "You should speak to your manager about the situation and explain your "
    "concerns clearly. It may help to describe what happened, how the issue "
    "affected your work, and what support you need. Before the meeting, write "
    "down the main points you want to discuss so that you can communicate "
    "them calmly and accurately. You should also keep a record of important "
    "messages and relevant events. If the problem continues, you may want to "
    "ask another appropriate person for guidance about the available options."
)


# ------------------------------------------------------------
# 1. Generate all eight conditions
# ------------------------------------------------------------

variants_a = generate_noise_variants(
    synthetic_text,
    source_index="synthetic",
    pair_index=1,
    base_seed=RANDOM_SEED,
)

assert set(variants_a) == set(NOISE_CONDITIONS)
assert len(variants_a) == 8

print("Noise conditions generated:", len(variants_a))


# ------------------------------------------------------------
# 2. Clean condition must be unchanged
# ------------------------------------------------------------

assert variants_a["clean"] == synthetic_text

print("Clean condition preservation passed.")


# ------------------------------------------------------------
# 3. Determinism
# ------------------------------------------------------------

variants_b = generate_noise_variants(
    synthetic_text,
    source_index="synthetic",
    pair_index=1,
    base_seed=RANDOM_SEED,
)

assert variants_a == variants_b

print("Determinism check passed.")


# ------------------------------------------------------------
# 4. Each single-noise condition should perturb this sufficiently
#    long synthetic response
# ------------------------------------------------------------

for condition in (
    "casing",
    "punctuation",
    "whitespace",
    "typo",
    "word_deletion",
    "filler",
    "mixed",
):
    assert variants_a[condition] != synthetic_text, (
        f"{condition} did not alter the synthetic test text"
    )

print("Synthetic perturbation checks passed.")


# ------------------------------------------------------------
# 5. Word deletion must retain at least 90% of alphabetic tokens
# ------------------------------------------------------------

original_words = re.findall(
    r"\b[A-Za-z]+\b",
    synthetic_text,
)

deleted_words = re.findall(
    r"\b[A-Za-z]+\b",
    variants_a["word_deletion"],
)

retention_rate = (
    len(deleted_words)
    / len(original_words)
)

assert retention_rate >= 0.90
assert len(deleted_words) < len(original_words)

print(
    "Word-deletion retention:",
    round(retention_rate, 3),
)


# ------------------------------------------------------------
# 6. Empty text should remain safe
# ------------------------------------------------------------

empty_variants = generate_noise_variants(
    "",
    source_index="empty",
    pair_index=0,
    base_seed=RANDOM_SEED,
)

assert all(
    value == ""
    for value in empty_variants.values()
)

print("Empty-input handling passed.")


# ------------------------------------------------------------
# 7. Invalid conditions must fail explicitly
# ------------------------------------------------------------

try:
    apply_noise_condition(
        synthetic_text,
        "invalid_condition",
        example_key="synthetic:1",
        base_seed=RANDOM_SEED,
    )

except ValueError:
    invalid_condition_check = True

else:
    invalid_condition_check = False

assert invalid_condition_check

print("Invalid-condition handling passed.")


print("\nnoise_v1 synthetic tests passed.")

In [ ]:
noise_records = []

for _, row in pilot_df.iterrows():
    variants = generate_noise_variants(
        row["assistant_text"],
        source_index=row["source_index"],
        pair_index=row["pair_index"],
        base_seed=RANDOM_SEED,
    )

    for condition, condition_text in variants.items():
        record = row.to_dict()

        record["assistant_text_clean"] = row["assistant_text"]
        record["noise_condition"] = condition
        record["assistant_text_condition"] = condition_text
        record["noise_version"] = NOISE_VERSION

        noise_records.append(record)


noise_pilot_df = pd.DataFrame(noise_records)

print("Generated rows:", len(noise_pilot_df))
print(
    "Unique source conversations:",
    noise_pilot_df["source_index"].nunique(),
)
print(
    "Noise conditions:",
    noise_pilot_df["noise_condition"].nunique(),
)
print(
    "Noise version:",
    noise_pilot_df["noise_version"].unique().tolist(),
)

In [ ]:
# ------------------------------------------------------------
# Structural integrity checks
# ------------------------------------------------------------

assert len(noise_pilot_df) == 800

assert (
    noise_pilot_df["source_index"].nunique()
    == 100
)

assert (
    noise_pilot_df["noise_condition"].nunique()
    == 8
)

assert (
    noise_pilot_df["noise_version"]
    .eq("noise_v1")
    .all()
)

assert (
    noise_pilot_df["assistant_text_clean"]
    .notna()
    .all()
)

assert (
    noise_pilot_df["assistant_text_condition"]
    .notna()
    .all()
)


# Every original example must have exactly all eight conditions.
condition_sets = (
    noise_pilot_df
    .groupby(
        ["source_index", "pair_index"]
    )["noise_condition"]
    .apply(set)
)

expected_conditions = set(
    NOISE_CONDITIONS
)

assert condition_sets.apply(
    lambda values: values == expected_conditions
).all()


# Clean-condition text must be identical to the original assistant text.
clean_rows = noise_pilot_df[
    noise_pilot_df["noise_condition"] == "clean"
]

assert (
    clean_rows["assistant_text_condition"]
    == clean_rows["assistant_text_clean"]
).all()


# Weak labels must remain unchanged across all noise variants.
label_columns = [
    "sycophantic_validation",
    "overconfident_judgement",
    "directive_advice",
    "weak_label_any",
    "weak_label_count",
    "weak_labels",
]

for column in label_columns:
    per_example_values = (
        noise_pilot_df
        .groupby(
            ["source_index", "pair_index"]
        )[column]
        .nunique(dropna=False)
    )

    assert (
        per_example_values == 1
    ).all(), f"Label changed across conditions: {column}"


print("800-row noise dataset integrity checks passed.")

In [ ]:
# ------------------------------------------------------------
# Noise-condition change-rate diagnostics
# ------------------------------------------------------------

noise_pilot_df["text_changed"] = (
    noise_pilot_df["assistant_text_condition"]
    != noise_pilot_df["assistant_text_clean"]
)

change_summary = (
    noise_pilot_df
    .groupby("noise_condition")
    .agg(
        rows=("text_changed", "size"),
        changed=("text_changed", "sum"),
    )
    .reindex(NOISE_CONDITIONS)
)

change_summary["unchanged"] = (
    change_summary["rows"]
    - change_summary["changed"]
)

change_summary["changed_pct"] = (
    change_summary["changed"]
    / change_summary["rows"]
    * 100
).round(1)


# Clean text must never change.
assert int(
    change_summary.loc["clean", "changed"]
) == 0


# Every synthetic noise family must change at least some real examples.
for condition in NOISE_CONDITIONS:
    if condition == "clean":
        continue

    assert int(
        change_summary.loc[condition, "changed"]
    ) > 0, (
        f"No pilot examples were changed by {condition}"
    )


# Noise generation must never produce an empty response.
assert (
    noise_pilot_df["assistant_text_condition"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)


print("Noise-condition change rates:")
change_summary

In [ ]:
# ------------------------------------------------------------
# Realised noise-severity diagnostics
# ------------------------------------------------------------

def count_alpha_words(text):
    return len(
        re.findall(
            r"\b[A-Za-z]+\b",
            str(text),
        )
    )


severity_df = noise_pilot_df.copy()

severity_df["clean_char_count"] = (
    severity_df["assistant_text_clean"]
    .astype(str)
    .str.len()
)

severity_df["condition_char_count"] = (
    severity_df["assistant_text_condition"]
    .astype(str)
    .str.len()
)

severity_df["clean_word_count"] = (
    severity_df["assistant_text_clean"]
    .apply(count_alpha_words)
)

severity_df["condition_word_count"] = (
    severity_df["assistant_text_condition"]
    .apply(count_alpha_words)
)


severity_df["char_length_ratio"] = (
    severity_df["condition_char_count"]
    / severity_df["clean_char_count"]
)

severity_df["word_count_ratio"] = (
    severity_df["condition_word_count"]
    / severity_df["clean_word_count"]
)


severity_summary = (
    severity_df
    .groupby("noise_condition")
    .agg(
        mean_char_ratio=("char_length_ratio", "mean"),
        min_char_ratio=("char_length_ratio", "min"),
        max_char_ratio=("char_length_ratio", "max"),
        mean_word_ratio=("word_count_ratio", "mean"),
        min_word_ratio=("word_count_ratio", "min"),
        max_word_ratio=("word_count_ratio", "max"),
    )
    .reindex(NOISE_CONDITIONS)
    .round(3)
)


# ------------------------------------------------------------
# Word-deletion safeguard
# ------------------------------------------------------------

word_deletion_rows = severity_df[
    severity_df["noise_condition"] == "word_deletion"
].copy()

assert (
    word_deletion_rows["word_count_ratio"]
    >= 0.90
).all(), (
    "At least one word-deletion example retained "
    "less than 90% of its original words."
)


# Clean condition must remain exactly unchanged.
clean_severity_rows = severity_df[
    severity_df["noise_condition"] == "clean"
]

assert (
    clean_severity_rows["char_length_ratio"] == 1.0
).all()

assert (
    clean_severity_rows["word_count_ratio"] == 1.0
).all()


print("Realised noise-severity summary:")
severity_summary

In [ ]:
print(
    "Minimum word retention under word deletion:",
    round(
        word_deletion_rows["word_count_ratio"].min(),
        3,
    ),
)

print(
    "Mean word retention under word deletion:",
    round(
        word_deletion_rows["word_count_ratio"].mean(),
        3,
    ),
)

print("Realised noise-severity checks passed.")

In [ ]:
# ------------------------------------------------------------
# Qualitative noise spot-check
# ------------------------------------------------------------

positive_examples = (
    noise_pilot_df[
        noise_pilot_df["weak_label_any"] == True
    ][
        [
            "source_index",
            "pair_index",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["source_index", "pair_index"]
    )
    .head(2)
)

negative_examples = (
    noise_pilot_df[
        noise_pilot_df["weak_label_any"] == False
    ][
        [
            "source_index",
            "pair_index",
        ]
    ]
    .drop_duplicates()
    .sample(
        n=2,
        random_state=RANDOM_SEED,
    )
)

spotcheck_examples = pd.concat(
    [
        positive_examples,
        negative_examples,
    ],
    ignore_index=True,
)

spotcheck_df = noise_pilot_df.merge(
    spotcheck_examples,
    on=[
        "source_index",
        "pair_index",
    ],
    how="inner",
)

spotcheck_df = spotcheck_df[
    [
        "source_index",
        "pair_index",
        "weak_label_any",
        "noise_condition",
        "assistant_text_condition",
    ]
].copy()


# Keep notebook output compact.
spotcheck_df["text_preview"] = (
    spotcheck_df["assistant_text_condition"]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.slice(
        0,
        300,
    )
)

spotcheck_output = (
    spotcheck_df[
        [
            "source_index",
            "pair_index",
            "weak_label_any",
            "noise_condition",
            "text_preview",
        ]
    ]
    .sort_values(
        [
            "source_index",
            "pair_index",
            "noise_condition",
        ]
    )
    .reset_index(drop=True)
)

spotcheck_output

## Noise-generation validation outcome

The `noise_v1` configuration was validated before preprocessing.

Validation included:

- synthetic tests covering all eight experimental conditions;
- deterministic repeat-generation checks;
- preservation of the original clean condition;
- explicit handling of empty and invalid inputs;
- structural checks across the 800 generated pilot-condition records;
- realised perturbation-rate diagnostics;
- a minimum 90% word-retention safeguard for the dedicated word-deletion
  condition; and
- qualitative inspection of weak-label-positive and weak-label-negative
  examples across all noise conditions.

On the 100-example pilot, the proportion of responses changed by each
single-noise condition ranged from 94% to 100%. The mixed condition changed
all 100 responses.

For the dedicated word-deletion condition:

- minimum observed alphabetic-word retention was `0.968`;
- mean observed alphabetic-word retention was `0.981`.

The transformations remained recognisable as variants of their corresponding
clean responses during qualitative inspection. The mixed condition was more
challenging than the individual conditions but did not produce widespread
unreadable or empty text.

`noise_v1` is therefore frozen for the pilot experiment. Subsequent changes to
the noise-generation rules or rates would require a new version rather than
silent modification of this configuration.

These diagnostics describe the experimental pilot only and are not estimates
of naturally occurring noise in LMSYS-Chat-1M.

## Preprocessing specification

Preprocessing is applied only to the text held in
`assistant_text_condition`. Source identifiers, user text, manual relevance
annotations, weak labels, and the original clean assistant response remain
unchanged.

Four preprocessing configurations are used. They deliberately range from no
cleaning to relatively aggressive cleaning so that the experiment can measure
both robustness gains and possible loss of linguistically important cues.

### 1. `none`

No preprocessing is applied.

The input text is returned exactly as received.

This condition provides the reference against which the cleaning strategies
are compared.

### 2. `minimal`

Minimal normalization performs only low-risk formatting cleanup:

- Unicode normalization using NFKC;
- normalization of line breaks and tabs to spaces;
- collapse repeated whitespace;
- remove leading and trailing whitespace.

It preserves:

- original word choice;
- capitalization;
- punctuation;
- modal expressions;
- pronouns;
- discourse markers.

This configuration is intended to remove formatting noise while making as few
linguistic changes as possible.

### 3. `noise_aware`

Noise-aware preprocessing extends `minimal` preprocessing with transformations
targeted at several synthetic noise families:

- lowercase alphabetic text;
- normalize repeated `.`, `!`, and `?` punctuation;
- normalize spacing immediately before punctuation;
- normalize spacing after punctuation where appropriate;
- remove clearly identifiable filler tokens `um` and `uh`.

Ambiguous discourse expressions such as `well` and `you know` are retained
because they may occur naturally and should not automatically be treated as
noise.

This configuration does not attempt automatic spelling correction or recovery
of words joined by deleted whitespace. These operations can be ambiguous and
could introduce text that was not present in the source response.

### 4. `aggressive`

Aggressive preprocessing extends `noise_aware` preprocessing with:

- removal of remaining punctuation and non-alphanumeric symbols;
- removal of standard English stop words;
- final whitespace normalization.

This configuration intentionally represents a stronger traditional NLP
cleaning strategy.

Stop-word removal may delete pronouns or modal constructions that contribute
to disempowerment cues. For example, expressions involving terms such as
`you`, `should`, or `must` may be altered. This is therefore not assumed to be
the best preprocessing strategy. Its inclusion allows the experiment to
measure whether aggressive cleaning damages task-relevant linguistic
information.

## Experimental preprocessing matrix

Each of the 800 clean/noisy condition records will be processed using all four
configurations:

1. `none`
2. `minimal`
3. `noise_aware`
4. `aggressive`

This produces:

`800 noise-condition records × 4 preprocessing configurations = 3,200 records`

The same source conversation and weak label remain associated with every
derived variant.

The experiment therefore separates two factors:

- **noise condition** — what corruption was introduced; and
- **preprocessing configuration** — what cleaning was subsequently applied.

All preprocessing will be deterministic and versioned as `preprocess_v1`.

The aggressive strategy is included specifically to evaluate potential
information loss rather than because aggressive cleaning is expected to
improve classifier performance.

In [ ]:
from src.preprocessing import (
    PREPROCESS_VERSION,
    PREPROCESS_CONFIGS,
    apply_preprocessing,
    generate_preprocessing_variants,
)


# ------------------------------------------------------------
# 1. Version and configuration checks
# ------------------------------------------------------------

assert PREPROCESS_VERSION == "preprocess_v1"

assert PREPROCESS_CONFIGS == (
    "none",
    "minimal",
    "noise_aware",
    "aggressive",
)

print("Preprocessing configuration check passed.")


# ------------------------------------------------------------
# 2. Synthetic noisy example
# ------------------------------------------------------------

synthetic_noisy_text = (
    "  YOU  should, um, SPEAK to your manager!!!\n"
    "It  is important to explain your concerns???\t"
    "Uh, you should also keep a record of what happened.  "
)


variants = generate_preprocessing_variants(
    synthetic_noisy_text
)

assert set(variants) == set(PREPROCESS_CONFIGS)

print("Generated preprocessing variants:", len(variants))


# ------------------------------------------------------------
# 3. 'none' must preserve the input exactly
# ------------------------------------------------------------

assert (
    variants["none"]
    == synthetic_noisy_text
)

print("'none' preservation check passed.")


# ------------------------------------------------------------
# 4. Minimal preprocessing
# ------------------------------------------------------------

minimal_text = variants["minimal"]

assert "\n" not in minimal_text
assert "\t" not in minimal_text
assert "  " not in minimal_text

# Minimal preprocessing must preserve casing and repeated punctuation.
assert "YOU" in minimal_text
assert "SPEAK" in minimal_text
assert "!!!" in minimal_text
assert "???" in minimal_text

# Minimal preprocessing must preserve fillers.
assert "um" in minimal_text.lower()
assert "uh" in minimal_text.lower()

print("Minimal preprocessing checks passed.")


# ------------------------------------------------------------
# 5. Noise-aware preprocessing
# ------------------------------------------------------------

noise_aware_text = variants["noise_aware"]

# Text should be lowercased.
assert noise_aware_text == noise_aware_text.lower()

# Repeated punctuation should be normalized.
assert "!!!" not in noise_aware_text
assert "???" not in noise_aware_text
assert "!" in noise_aware_text
assert "?" in noise_aware_text

# Explicit fillers should be removed.
assert not re.search(
    r"\b(?:um|uh)\b",
    noise_aware_text,
)

# Linguistically meaningful words should remain.
assert "you should" in noise_aware_text
assert "manager" in noise_aware_text
assert "important" in noise_aware_text

print("Noise-aware preprocessing checks passed.")


# ------------------------------------------------------------
# 6. Aggressive preprocessing
# ------------------------------------------------------------

aggressive_text = variants["aggressive"]

# Punctuation should be removed.
assert not re.search(
    r"[^a-z0-9\s]",
    aggressive_text,
)

# No repeated whitespace should remain.
assert "  " not in aggressive_text

# Aggressive cleaning should remove at least some common stop words.
noise_aware_tokens = re.findall(
    r"\b[a-z0-9]+\b",
    noise_aware_text,
)

aggressive_tokens = re.findall(
    r"\b[a-z0-9]+\b",
    aggressive_text,
)

assert (
    len(aggressive_tokens)
    < len(noise_aware_tokens)
)

# Content-bearing words should remain.
assert "manager" in aggressive_tokens
assert "important" in aggressive_tokens
assert "concerns" in aggressive_tokens

print("Aggressive preprocessing checks passed.")


# ------------------------------------------------------------
# 7. Determinism
# ------------------------------------------------------------

variants_repeat = generate_preprocessing_variants(
    synthetic_noisy_text
)

assert variants == variants_repeat

print("Determinism check passed.")


# ------------------------------------------------------------
# 8. Empty-input handling
# ------------------------------------------------------------

empty_variants = generate_preprocessing_variants(
    ""
)

assert all(
    value == ""
    for value in empty_variants.values()
)

print("Empty-input handling passed.")


# ------------------------------------------------------------
# 9. Invalid configuration handling
# ------------------------------------------------------------

try:
    apply_preprocessing(
        synthetic_noisy_text,
        "invalid_config",
    )

except ValueError:
    invalid_config_check = True

else:
    invalid_config_check = False

assert invalid_config_check

print("Invalid-configuration handling passed.")


# ------------------------------------------------------------
# 10. Display compact synthetic comparison
# ------------------------------------------------------------

for config in PREPROCESS_CONFIGS:
    print(f"\n[{config}]")
    print(variants[config])


print("\npreprocess_v1 synthetic tests passed.")

In [ ]:
# ------------------------------------------------------------
# Generate preprocessing variants for all noise-condition rows
# ------------------------------------------------------------

preprocess_records = []

for _, row in noise_pilot_df.iterrows():
    variants = generate_preprocessing_variants(
        row["assistant_text_condition"]
    )

    for config, processed_text in variants.items():
        record = row.to_dict()

        record["preprocess_config"] = config
        record["assistant_text_processed"] = processed_text
        record["preprocess_version"] = PREPROCESS_VERSION

        preprocess_records.append(
            record
        )


experiment_df = pd.DataFrame(
    preprocess_records
)

print(
    "Generated rows:",
    len(experiment_df),
)

print(
    "Unique source conversations:",
    experiment_df["source_index"].nunique(),
)

print(
    "Noise conditions:",
    experiment_df["noise_condition"].nunique(),
)

print(
    "Preprocessing configurations:",
    experiment_df["preprocess_config"].nunique(),
)

print(
    "Preprocessing version:",
    experiment_df["preprocess_version"].unique().tolist(),
)

In [ ]:
# ------------------------------------------------------------
# Experimental-matrix structural integrity checks
# ------------------------------------------------------------

assert len(experiment_df) == 3200

assert (
    experiment_df["source_index"].nunique()
    == 100
)

assert (
    experiment_df["noise_condition"].nunique()
    == 8
)

assert (
    experiment_df["preprocess_config"].nunique()
    == 4
)

assert (
    experiment_df["noise_version"]
    .eq("noise_v1")
    .all()
)

assert (
    experiment_df["preprocess_version"]
    .eq("preprocess_v1")
    .all()
)

assert (
    experiment_df["assistant_text_processed"]
    .notna()
    .all()
)


# Every source pair must contain all 8 × 4 combinations.
matrix_counts = (
    experiment_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )
    .size()
)

assert (
    matrix_counts == 32
).all()


# Every noise condition must have all four preprocessing configs.
config_sets = (
    experiment_df
    .groupby(
        [
            "source_index",
            "pair_index",
            "noise_condition",
        ]
    )["preprocess_config"]
    .apply(set)
)

assert config_sets.apply(
    lambda values:
    values == set(PREPROCESS_CONFIGS)
).all()


# 'none' preprocessing must preserve the condition text exactly.
none_rows = experiment_df[
    experiment_df["preprocess_config"]
    == "none"
]

assert (
    none_rows["assistant_text_processed"]
    == none_rows["assistant_text_condition"]
).all()


# Weak labels must remain invariant across all 32 variants
# of each original source example.
label_columns = [
    "sycophantic_validation",
    "overconfident_judgement",
    "directive_advice",
    "weak_label_any",
    "weak_label_count",
    "weak_labels",
]

for column in label_columns:
    values_per_source = (
        experiment_df
        .groupby(
            [
                "source_index",
                "pair_index",
            ]
        )[column]
        .nunique(
            dropna=False
        )
    )

    assert (
        values_per_source == 1
    ).all(), (
        f"Label changed across derived variants: {column}"
    )


print(
    "3,200-row experimental matrix integrity checks passed."
)

In [ ]:
# ------------------------------------------------------------
# Preprocessing transformation diagnostics
# ------------------------------------------------------------

experiment_df["preprocess_changed"] = (
    experiment_df["assistant_text_processed"]
    != experiment_df["assistant_text_condition"]
)

preprocess_change_summary = (
    experiment_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .agg(
        rows=("preprocess_changed", "size"),
        changed=("preprocess_changed", "sum"),
    )
    .reset_index()
)

preprocess_change_summary["unchanged"] = (
    preprocess_change_summary["rows"]
    - preprocess_change_summary["changed"]
)

preprocess_change_summary["changed_pct"] = (
    preprocess_change_summary["changed"]
    / preprocess_change_summary["rows"]
    * 100
).round(1)


# Put conditions/configurations in experimental order.
preprocess_change_summary["noise_condition"] = pd.Categorical(
    preprocess_change_summary["noise_condition"],
    categories=NOISE_CONDITIONS,
    ordered=True,
)

preprocess_change_summary["preprocess_config"] = pd.Categorical(
    preprocess_change_summary["preprocess_config"],
    categories=PREPROCESS_CONFIGS,
    ordered=True,
)

preprocess_change_summary = (
    preprocess_change_summary
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


# 'none' must never alter its input.
none_change_rows = preprocess_change_summary[
    preprocess_change_summary["preprocess_config"]
    == "none"
]

assert (
    none_change_rows["changed"] == 0
).all()


print("Preprocessing change rates:")
preprocess_change_summary

In [ ]:
# ------------------------------------------------------------
# Preprocessing information-reduction diagnostics
# ------------------------------------------------------------

def count_processed_words(text):
    return len(
        re.findall(
            r"\b[A-Za-z0-9]+\b",
            str(text),
        )
    )


reduction_df = experiment_df.copy()

reduction_df["input_char_count"] = (
    reduction_df["assistant_text_condition"]
    .astype(str)
    .str.len()
)

reduction_df["processed_char_count"] = (
    reduction_df["assistant_text_processed"]
    .astype(str)
    .str.len()
)

reduction_df["input_word_count"] = (
    reduction_df["assistant_text_condition"]
    .apply(count_processed_words)
)

reduction_df["processed_word_count"] = (
    reduction_df["assistant_text_processed"]
    .apply(count_processed_words)
)


reduction_df["char_retention_ratio"] = (
    reduction_df["processed_char_count"]
    / reduction_df["input_char_count"]
)

reduction_df["word_retention_ratio"] = (
    reduction_df["processed_word_count"]
    / reduction_df["input_word_count"]
)


reduction_summary = (
    reduction_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .agg(
        mean_char_retention=(
            "char_retention_ratio",
            "mean",
        ),
        min_char_retention=(
            "char_retention_ratio",
            "min",
        ),
        mean_word_retention=(
            "word_retention_ratio",
            "mean",
        ),
        min_word_retention=(
            "word_retention_ratio",
            "min",
        ),
    )
    .reset_index()
)


reduction_summary["noise_condition"] = pd.Categorical(
    reduction_summary["noise_condition"],
    categories=NOISE_CONDITIONS,
    ordered=True,
)

reduction_summary["preprocess_config"] = pd.Categorical(
    reduction_summary["preprocess_config"],
    categories=PREPROCESS_CONFIGS,
    ordered=True,
)

reduction_summary = (
    reduction_summary
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)

for column in [
    "mean_char_retention",
    "min_char_retention",
    "mean_word_retention",
    "min_word_retention",
]:
    reduction_summary[column] = (
        reduction_summary[column]
        .round(3)
    )


# 'none' must retain the input exactly.
none_reduction = reduction_summary[
    reduction_summary["preprocess_config"]
    == "none"
]

assert (
    none_reduction["mean_char_retention"]
    == 1.0
).all()

assert (
    none_reduction["mean_word_retention"]
    == 1.0
).all()


# No preprocessing configuration may produce empty responses.
assert (
    reduction_df["assistant_text_processed"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)


print("Preprocessing retention summary:")
reduction_summary

In [ ]:
from src.labelling import (
    WEAK_LABEL_VERSION,
    weak_label_disempowerment,
)


# ------------------------------------------------------------
# Cue-preservation diagnostic for the original directive positives
# ------------------------------------------------------------

assert WEAK_LABEL_VERSION == "disempowerment_weak_v5"


# Restrict this diagnostic to the six examples that were labelled
# directive_advice in the frozen clean pilot.
directive_cue_df = experiment_df[
    experiment_df["directive_advice"].eq(True)
].copy()


print(
    "Original directive-positive sources:",
    directive_cue_df[
        ["source_index", "pair_index"]
    ]
    .drop_duplicates()
    .shape[0],
)

assert (
    directive_cue_df[
        ["source_index", "pair_index"]
    ]
    .drop_duplicates()
    .shape[0]
    == 6
)


# Re-run the frozen weak-labelling rule on the processed assistant text.
# This is a cue-preservation diagnostic only; it does NOT create new
# ground-truth labels.
rechecked_labels = directive_cue_df.apply(
    lambda row: weak_label_disempowerment(
        user_text=row["user_text"],
        assistant_text=row["assistant_text_processed"],
    ),
    axis=1,
)


directive_cue_df[
    "directive_cue_detected_after_processing"
] = [
    result["directive_advice"]
    for result in rechecked_labels
]


# ------------------------------------------------------------
# Summarise cue survival for each noise × preprocessing combination
# ------------------------------------------------------------

cue_retention_summary = (
    directive_cue_df
    .groupby(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .agg(
        original_positive_examples=(
            "source_index",
            "size",
        ),
        directive_cue_detected=(
            "directive_cue_detected_after_processing",
            "sum",
        ),
    )
    .reset_index()
)


cue_retention_summary["cue_retention_pct"] = (
    cue_retention_summary[
        "directive_cue_detected"
    ]
    / cue_retention_summary[
        "original_positive_examples"
    ]
    * 100
).round(1)


cue_retention_summary["noise_condition"] = pd.Categorical(
    cue_retention_summary["noise_condition"],
    categories=NOISE_CONDITIONS,
    ordered=True,
)

cue_retention_summary["preprocess_config"] = pd.Categorical(
    cue_retention_summary["preprocess_config"],
    categories=PREPROCESS_CONFIGS,
    ordered=True,
)

cue_retention_summary = (
    cue_retention_summary
    .sort_values(
        [
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


# Each experimental cell must contain the same six original positives.
assert (
    cue_retention_summary[
        "original_positive_examples"
    ]
    == 6
).all()


# The untouched clean baseline must reproduce all six original
# directive-advice detections.
clean_none = cue_retention_summary[
    (
        cue_retention_summary["noise_condition"]
        == "clean"
    )
    &
    (
        cue_retention_summary["preprocess_config"]
        == "none"
    )
]

assert (
    int(
        clean_none[
            "directive_cue_detected"
        ].iloc[0]
    )
    == 6
)


print("Directive-cue retention by experimental condition:")
cue_retention_summary

In [ ]:
# ------------------------------------------------------------
# Inspect directive-cue losses
# ------------------------------------------------------------

cue_loss_df = directive_cue_df[
    directive_cue_df[
        "directive_cue_detected_after_processing"
    ].eq(False)
].copy()


cue_loss_summary = (
    cue_loss_df[
        [
            "source_index",
            "pair_index",
            "noise_condition",
            "preprocess_config",
            "directive_evidence",
            "assistant_text_processed",
        ]
    ]
    .sort_values(
        [
            "source_index",
            "pair_index",
            "noise_condition",
            "preprocess_config",
        ]
    )
    .reset_index(drop=True)
)


# Keep displayed text compact.
cue_loss_summary["processed_preview"] = (
    cue_loss_summary[
        "assistant_text_processed"
    ]
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.slice(
        0,
        350,
    )
)


cue_loss_output = cue_loss_summary[
    [
        "source_index",
        "pair_index",
        "noise_condition",
        "preprocess_config",
        "directive_evidence",
        "processed_preview",
    ]
]


print(
    "Total cue-loss variants:",
    len(cue_loss_output),
)

print(
    "Unique original positive examples affected:",
    cue_loss_output[
        ["source_index", "pair_index"]
    ]
    .drop_duplicates()
    .shape[0],
)

cue_loss_output

In [ ]:
# ------------------------------------------------------------
# Diagnose the single non-aggressive cue-loss example
# ------------------------------------------------------------

target_source = 43461
target_pair = 4

target_diagnostic = directive_cue_df[
    (directive_cue_df["source_index"] == target_source)
    & (directive_cue_df["pair_index"] == target_pair)
    & (directive_cue_df["preprocess_config"] == "none")
    & (
        directive_cue_df["noise_condition"].isin(
            [
                "clean",
                "whitespace",
                "word_deletion",
                "mixed",
            ]
        )
    )
].copy()


def extract_relevant_context(text, window=450):
    text = str(text)

    keyword_pattern = re.compile(
        r"appropriate|contact|reach\s+out|message|"
        r"text|call|blocked|wishes|boundary|decision",
        re.IGNORECASE,
    )

    match = keyword_pattern.search(text)

    if match is None:
        return text[:window]

    start = max(
        0,
        match.start() - 180,
    )

    end = min(
        len(text),
        match.end() + 270,
    )

    return text[start:end]


target_diagnostic["cue_context"] = (
    target_diagnostic[
        "assistant_text_processed"
    ]
    .apply(extract_relevant_context)
)

target_diagnostic = target_diagnostic[
    [
        "noise_condition",
        "directive_cue_detected_after_processing",
        "cue_context",
    ]
].sort_values(
    "noise_condition"
).reset_index(drop=True)

target_diagnostic

In [ ]:
# ------------------------------------------------------------
# Print exact cue region for source 43461
# ------------------------------------------------------------

target_rows = directive_cue_df[
    (directive_cue_df["source_index"] == 43461)
    & (directive_cue_df["pair_index"] == 4)
    & (directive_cue_df["preprocess_config"] == "none")
    & (
        directive_cue_df["noise_condition"].isin(
            [
                "clean",
                "whitespace",
                "word_deletion",
                "mixed",
            ]
        )
    )
].copy()


def cue_region(text):
    text = str(text)

    # Look for terms associated with the original
    # inappropriate-contact weak-label evidence.
    matches = list(
        re.finditer(
            r"appropriate|contact|reach\s*out|message|text|call",
            text,
            flags=re.IGNORECASE,
        )
    )

    if not matches:
        return "[No cue keyword found]"

    start = max(
        0,
        matches[0].start() - 120,
    )

    end = min(
        len(text),
        matches[-1].end() + 120,
    )

    return text[start:end]


for condition in [
    "clean",
    "whitespace",
    "word_deletion",
    "mixed",
]:
    row = target_rows[
        target_rows["noise_condition"] == condition
    ].iloc[0]

    print("=" * 80)
    print("CONDITION:", condition)
    print(
        "CUE DETECTED:",
        row["directive_cue_detected_after_processing"],
    )
    print()
    print(
        cue_region(
            row["assistant_text_processed"]
        )
    )
    print()

## Linguistic cue-preservation diagnostic

The frozen `disempowerment_weak_v5` rules were reapplied to the processed text
of the six original directive-advice-positive pilot examples as a diagnostic of
linguistic cue preservation.

This procedure did not alter the original weak labels. It measured whether the
same transparent linguistic cues remained detectable after noise injection and
preprocessing.

The clean, casing, punctuation, typo, and filler conditions retained all six
directive cues under `none`, `minimal`, and `noise_aware` preprocessing.

Whitespace corruption, word deletion, and mixed noise retained five of the six
original directive cues (`83.3%`) under these non-aggressive preprocessing
configurations.

Inspection showed that all non-aggressive cue losses involved the same source
example, whose original evidence was the `inappropriate_contact` construction:

`It's not appropriate to try to contact her ...`

The relevant surface form was disrupted in different ways:

- whitespace noise joined `to` and `try` (`totry`);
- word deletion removed the first `to`;
- mixed noise joined part of the `appropriate to` construction.

The underlying response remained broadly interpretable, but the exact lexical
pattern required by the frozen weak-labelling rule was no longer present. These
cases are retained as genuine robustness effects rather than treated as
generation errors.

Aggressive preprocessing resulted in `0/6` directive cues being redetected
across every noise condition. This is consistent with its removal of
stop words, punctuation, pronouns, and modal/function-word constructions that
form part of the original directive cues.

These results are descriptive diagnostics from six pilot positives. They are
not classifier-performance results and should not be generalized beyond the
pilot at this stage.

In [ ]:
# ------------------------------------------------------------
# Save frozen experimental matrix
# ------------------------------------------------------------

experiment_matrix_path = (
    PROJECT_ROOT
    / "data"
    / "samples"
    / "lmsys_noise_preprocess_v1_matrix_3200.csv"
)

experiment_df.to_csv(
    experiment_matrix_path,
    index=False,
)

print("Saved experimental matrix:")
print(experiment_matrix_path)
print("File exists:", experiment_matrix_path.exists())

In [ ]:
# ------------------------------------------------------------
# Reload and independently verify saved matrix
# ------------------------------------------------------------

saved_experiment_df = pd.read_csv(
    experiment_matrix_path
)

assert len(saved_experiment_df) == 3200

assert (
    saved_experiment_df["source_index"].nunique()
    == 100
)

assert (
    saved_experiment_df["noise_condition"].nunique()
    == 8
)

assert (
    saved_experiment_df["preprocess_config"].nunique()
    == 4
)

assert (
    saved_experiment_df["noise_version"]
    .eq("noise_v1")
    .all()
)

assert (
    saved_experiment_df["preprocess_version"]
    .eq("preprocess_v1")
    .all()
)

assert (
    saved_experiment_df["weak_label_version"]
    .eq("disempowerment_weak_v5")
    .all()
)

assert (
    saved_experiment_df["assistant_text_processed"]
    .notna()
    .all()
)


# 32 derived variants for every source pair.
saved_counts = (
    saved_experiment_df
    .groupby(
        [
            "source_index",
            "pair_index",
        ]
    )
    .size()
)

assert (
    saved_counts == 32
).all()


# Confirm original frozen weak-label distribution.
source_labels = (
    saved_experiment_df[
        [
            "source_index",
            "pair_index",
            "directive_advice",
            "sycophantic_validation",
            "overconfident_judgement",
        ]
    ]
    .drop_duplicates(
        [
            "source_index",
            "pair_index",
        ]
    )
)

assert len(source_labels) == 100

assert int(
    source_labels["directive_advice"].sum()
) == 6

assert int(
    source_labels["sycophantic_validation"].sum()
) == 0

assert int(
    source_labels["overconfident_judgement"].sum()
) == 0


print("Saved experimental matrix verification passed.")
print("Rows:", len(saved_experiment_df))
print(
    "Derived variants per source:",
    int(saved_counts.iloc[0]),
)

## Final pilot outcome

The noise-injection and preprocessing stage is complete for the 100-example
weak-labelled pilot.

Final configuration:

- Noise version: `noise_v1`
- Preprocessing version: `preprocess_v1`
- Source conversations: 100
- Noise conditions: 8
- Preprocessing configurations: 4
- Derived experimental records: 3,200

The frozen noise conditions are:

1. `clean`
2. `casing`
3. `punctuation`
4. `whitespace`
5. `typo`
6. `word_deletion`
7. `filler`
8. `mixed`

The frozen preprocessing configurations are:

1. `none`
2. `minimal`
3. `noise_aware`
4. `aggressive`

Validation completed during this notebook included:

- deterministic synthetic noise tests;
- deterministic preprocessing tests;
- structural integrity checks for the 800-row noise dataset;
- structural integrity checks for the 3,200-row experimental matrix;
- realised noise-severity diagnostics;
- preprocessing change-rate diagnostics;
- character- and word-retention diagnostics;
- qualitative inspection of representative noisy examples;
- directive-cue preservation analysis across the six frozen
  directive-advice-positive pilot examples; and
- independent reload verification of the saved experimental matrix.

The cue-preservation analysis showed that limited surface corruption could
prevent exact weak-labelling patterns from being redetected even where the
underlying response remained interpretable. Aggressive preprocessing removed
all six original directive cues under the frozen rule-based diagnostic,
consistent with its removal of function words and other task-relevant
linguistic structure.

These observations are pilot diagnostics only. They are not classifier
performance results and are not estimates of population-level prevalence.

The final experimental matrix was saved locally to:

`data/samples/lmsys_noise_preprocess_v1_matrix_3200.csv`

This matrix will provide the controlled clean, noisy, and cleaned-noisy text
conditions for the subsequent modelling and robustness-evaluation stage.